In [1]:
import os
import sys

if "google.colab" in sys.modules:
    # Running in Colab

    !git clone https://github.com/pthengtr/kcw-analytics.git
    !cd /content/kcw-analytics && git pull origin main

    from google.colab import drive
    drive.mount("/content/drive")

    BASE_FOLDER = "/content/drive/Shareddrives"
    BASE_FOLDER_GIT = "/content"
else:
    # Running in local Jupyter
    BASE_FOLDER = r"G:\Shared drives"
    BASE_FOLDER_GIT = r"C:\Users\Windows 11\Notebook"

print("Using folder:", BASE_FOLDER)

Cloning into 'kcw-analytics'...
remote: Enumerating objects: 823, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 823 (delta 123), reused 64 (delta 36), pack-reused 641 (from 2)
Receiving objects: 100% (823/823), 692.06 KiB | 18.21 MiB/s, done.
Resolving deltas: 100% (556/556), done.
From https://github.com/pthengtr/kcw-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Mounted at /content/drive
Using folder: /content/drive/Shareddrives


In [2]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw"

In [3]:
import os
import pandas as pd

data = {}

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)
        data[file] = pd.read_csv(
            path,
            dtype={
              "BCODE": "string",
              "MOBILE": "string",
              "BILLNO": "string",
            },
            encoding="utf-8-sig",
            low_memory=False   # stops chunk guessing
        )
        print(f"Loaded: {file} -> {data[file].shape}")

Loaded: raw_inventory_hq_2024.csv -> (4983, 8)
Loaded: raw_syp_pimas_purchase_bills.csv -> (3012, 49)
Loaded: raw_syp_simas_sales_bills.csv -> (13202, 49)
Loaded: raw_syp_pidet_purchase_lines.csv -> (28103, 41)
Loaded: raw_syp_sidet_sales_lines.csv -> (38554, 38)
Loaded: raw_hq_icmas_products.csv -> (114984, 94)
Loaded: raw_hq_pidet_purchase_lines.csv -> (154142, 41)
Loaded: raw_hq_pimas_purchase_bills.csv -> (50262, 49)
Loaded: raw_hq_sidet_sales_lines.csv -> (733804, 38)
Loaded: raw_hq_apmas_payable.csv -> (979, 20)
Loaded: raw_hq_armas_receivable.csv -> (2233, 20)
Loaded: raw_hq_simas_sales_bills.csv -> (276222, 49)
Loaded: raw_hq_pvmas_notes_vouchers.csv -> (13893, 32)
Loaded: raw_hq_rvmas_notes_vouchers.csv -> (13893, 32)


In [4]:
folder = f"{BASE_FOLDER}/KCW-Data/kcw_analytics/01_raw/online/tiktok"

In [22]:
import pandas as pd
import glob
import os

files = glob.glob(os.path.join(folder, "*.xlsx"))

df_tiktok_raw = pd.concat(
    (
        pd.read_excel(
            f,
            sheet_name="Order details",
            dtype={
              "Order/adjustment ID  ": "string",     # or the order id column
    }
        )
        for f in files
    ),
    ignore_index=True
)

In [23]:
df_tiktok_raw.columns

Index(['Order/adjustment ID  ', 'Type ', 'Order created time',
       'Order settled time', 'Currency', 'Total settlement amount',
       'Total Revenue', 'Subtotal after seller discounts',
       'Subtotal before discounts', 'Seller discounts',
       'Refund subtotal after seller discounts',
       'Refund subtotal before seller discounts', 'Refund of seller discounts',
       'Total Fees', 'Transaction fee', 'TikTok Shop commission fee',
       'Credit card installment - Interest rate cost', 'Seller shipping fee',
       'Actual shipping fee', 'Platform shipping fee discount',
       'Customer shipping fee', 'Actual return shipping fee',
       'Refunded customer shipping fee', 'Shipping subsidy',
       'Affiliate Commission',
       'Affiliate commission before PIT (personal income tax)',
       'Personal income tax withheld from affiliate commission',
       'Affiliate partner commission', 'Affiliate Shop Ads commission',
       'Affiliate Shop Ads commission before PIT',
       

In [24]:
df_tiktok_raw[['Order/adjustment ID  ','Total Revenue', 'Total settlement amount']]

,Order/adjustment ID,Total Revenue,Total settlement amount
0,582731372038948445,1395,1153.08
1,582725275153368212,1090,849.08
2,582722994427627285,875,731.48
3,582706281643541648,490,415.77
4,582738442743153703,340,280.96
...,...,...,...
138,582323000835671866,145,119.44
139,582318694736299256,380,315.93
140,582316901935646185,1090,847.35
141,582303499930076394,98,79.98


In [30]:
df = df_tiktok_raw.copy()

df = df.rename(columns={
    'Order/adjustment ID  ': 'tiktok_no',
    'Total Revenue': 'amount_full',
    'Total settlement amount': 'after_fee',
})

df["tiktok_no"] = (
    df["tiktok_no"]
    .astype("string")
    .str.strip()
    .str[3:]
)

df_tiktok_cleaned = df[["tiktok_no", "amount_full", "after_fee"]]

In [31]:
df_tiktok_cleaned

,tiktok_no,amount_full,after_fee
0,731372038948445,1395,1153.08
1,725275153368212,1090,849.08
2,722994427627285,875,731.48
3,706281643541648,490,415.77
4,738442743153703,340,280.96
...,...,...,...
138,323000835671866,145,119.44
139,318694736299256,380,315.93
140,316901935646185,1090,847.35
141,303499930076394,98,79.98


In [32]:
df_simas = data["raw_hq_simas_sales_bills.csv"].copy()

In [35]:
df_out = df_tiktok_cleaned.merge(
    df_simas[["PO", "BILLNO"]],
    how="left",
    left_on="tiktok_no",
    right_on="PO"
)

na_billno_cnt = df_out["BILLNO"].isna().sum()

df_out_clean = df_out.dropna(subset=["BILLNO"]).copy()

In [37]:
na_billno_cnt

np.int64(20)

In [38]:
df_out_clean["deduction_pct"] = (
    (df_out_clean["amount_full"] - df_out_clean["after_fee"]) / df_out_clean["amount_full"] * 100
).round(2)

In [39]:
df_out_clean

,tiktok_no,amount_full,after_fee,PO,BILLNO,deduction_pct
0,731372038948445,1395,1153.08,731372038948445,TAD6902-481,17.34
1,725275153368212,1090,849.08,725275153368212,TAD6902-479,22.10
2,722994427627285,875,731.48,722994427627285,TAD6902-478,16.40
3,706281643541648,490,415.77,706281643541648,TAD6902-475,15.15
4,738442743153703,340,280.96,738442743153703,TAD6902-505,17.36
...,...,...,...,...,...,...
138,323000835671866,145,119.44,323000835671866,TAD6901-589,17.63
139,318694736299256,380,315.93,318694736299256,TAD6901-588,16.86
140,316901935646185,1090,847.35,316901935646185,TAD6901-587,22.26
141,303499930076394,98,79.98,303499930076394,TAD6901-578,18.39


In [42]:
folder = os.path.join(
    BASE_FOLDER,
    "KCW-Data",
    "kcw_analytics",
    "03_curated",
    f"tiktok.csv"
)

# ⭐ create directory if missing
os.makedirs(os.path.dirname(folder), exist_ok=True)

In [43]:
df_out_clean[["BILLNO","deduction_pct"]].to_csv(folder, index=False, encoding="utf-8-sig")